# Denim Atlas — frozen calibration comparison

Private Kaggle GPU run. Upload only `denim-calibration-input.zip` as a private input, enable Internet and GPU. Contains licensed crop bytes and private rater judgments; do not publish notebook inputs/outputs. No training or held-out research claim. The bundle pins crop review, source notices, labels and the scoring source used for this run.

Outputs: frozen features, calibration-only B0/B1 report (w=0.5, fixed before labels), provenance hashes. Q1 is incomplete because its conflicting repeat is excluded, so no nDCG is emitted for it. Q2/Q3 can have descriptive calibration nDCG only. Save a version **with output files**, then stop the session.


In [ ]:
from pathlib import Path
import subprocess, sys, hashlib, json, zipfile
import torch
assert Path('/kaggle/working').is_dir(), 'Remote Kaggle only'
assert torch.cuda.is_available(), 'Enable GPU first; no CPU fallback'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','open_clip_torch==3.3.0'],check=True)


In [ ]:
expected_sha256='5b64231975691b7537a18f64e8e325a7d5e25032e546a2065d17bcb7e647a626'
archives=list(Path('/kaggle/input').rglob('denim-calibration-input.zip'))
assert len(archives)==1, 'Attach exactly one private input bundle'
archive=archives[0]
assert hashlib.sha256(archive.read_bytes()).hexdigest()==expected_sha256, 'Unexpected bundle bytes'
root=Path('/kaggle/working/calibration_round1_v1')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    assert len(z.namelist())==len(set(z.namelist())), 'Duplicate ZIP members'
    for member in z.infolist():
        target=(root/member.filename).resolve()
        assert target.is_relative_to(root.resolve()), 'Unsafe ZIP path'
    z.extractall(root)
subprocess.run([sys.executable,str(root/'scripts/extract_features.py'),
 '--manifest',str(root/'manifest.json'),'--image-root',str(root),
 '--output',str(root/'features.json')],check=True)
subprocess.run([sys.executable,str(root/'scripts/analyze_calibration.py'),
 '--packet',str(root/'packet.json'),'--ratings',str(root/'ratings.json'),
 '--features',str(root/'features.json'),'--output',str(root/'comparison.json')],check=True)
result=json.loads((root/'comparison.json').read_text())
assert result['purpose']=='calibration_only' and not result['held_out_evaluation']
print(json.dumps({'usable_pairs':result['usable_pairs'],'results':[
 {k:r[k] for k in ('baseline','query_id','fully_judged_pool','calibration_ndcg_at_5','top5')}
 for r in result['results']]},indent=2),flush=True)
